# 01 — Get the data

This notebook downloads the raw data used in the project and saves it in `data/raw/`:

- **Natural gas prices**: Henry Hub daily spot price (US Energy Information Administration, via FRED, series DHHNGSP). This file is not stored in the repository because of its licence; this notebook downloads it.
- **Temperatures**: daily mean temperature for 15 large US cities, 2008-01-01 to 2026-10-31 (ERA5 reanalysis, via the Open-Meteo API, licence CC BY 4.0).

Run this notebook first. It needs an internet connection and takes about 2 to 5 minutes, because the free Open-Meteo service limits the number of requests per minute.

In [1]:
from pathlib import Path
import time

import pandas as pd
import requests

# Find the project folder, wherever Python was started:
# in the repository itself, in code/, or one level above (as on Nuvolos).
here = Path.cwd()
candidates = [here, *here.parents, *[p for p in here.iterdir() if p.is_dir()]]
ROOT = next(p for p in candidates if (p / "code" / "01_get_data.ipynb").exists())

RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

print("Project folder:", ROOT)
print("Raw data folder:", RAW)

Project folder: /files/natgas-weather-volatility
Raw data folder: /files/natgas-weather-volatility/data/raw


In [2]:
# --- 1. Natural gas prices: Henry Hub daily spot price (EIA, via FRED) ---
PRICE_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DHHNGSP"

response = requests.get(PRICE_URL, timeout=60)
response.raise_for_status()
(RAW / "henry_hub_spot_daily.csv").write_bytes(response.content)

prices = pd.read_csv(RAW / "henry_hub_spot_daily.csv")
print("Prices:", prices.shape)
print(prices.head(3))
print(prices.tail(3))

Prices: (7791, 2)
  observation_date  DHHNGSP
0       1997-01-07     3.82
1       1997-01-08     3.80
2       1997-01-09     3.61
     observation_date  DHHNGSP
7788       2026-11-13     3.26
7789       2026-11-16     3.20
7790       2026-11-17     3.10


In [3]:
# --- 2. Temperatures: daily mean, 15 large US cities (ERA5 reanalysis, via Open-Meteo) ---
# The study period is fixed so that results do not change when the notebook is re-run later.
START, END = "2008-01-01", "2026-10-31"

CITIES = {
    "new_york":     (40.71, -74.01),
    "boston":       (42.36, -71.06),
    "philadelphia": (39.95, -75.17),
    "washington":   (38.91, -77.04),
    "pittsburgh":   (40.44, -80.00),
    "cleveland":    (41.50, -81.69),
    "columbus":     (39.96, -83.00),
    "detroit":      (42.33, -83.05),
    "indianapolis": (39.77, -86.16),
    "chicago":      (41.88, -87.63),
    "milwaukee":    (43.04, -87.91),
    "minneapolis":  (44.98, -93.27),
    "atlanta":      (33.75, -84.39),
    "dallas":       (32.78, -96.80),
    "houston":      (29.76, -95.37),
}

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"


def download_city(lat, lon, max_attempts=5):
    """Download daily mean temperatures for one location.
    The free Open-Meteo service limits requests per minute: if it answers
    '429 Too Many Requests', wait one minute and try again."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START,
        "end_date": END,
        "daily": "temperature_2m_mean",
        "timezone": "auto",
    }
    for attempt in range(1, max_attempts + 1):
        response = requests.get(OPEN_METEO_URL, params=params, timeout=60)
        if response.status_code == 429:
            print(f"    rate limit reached, waiting 65 s (attempt {attempt}/{max_attempts})")
            time.sleep(65)
            continue
        response.raise_for_status()
        return response.json()["daily"]
    raise RuntimeError("Open-Meteo still refuses after several attempts: try again later.")


tables = []
for city, (lat, lon) in CITIES.items():
    daily = download_city(lat, lon)
    tables.append(pd.DataFrame({"date": daily["time"], "city": city, "temp_c": daily["temperature_2m_mean"]}))
    print(f"{city:<13} {len(tables[-1])} days")
    time.sleep(5)

temperatures = pd.concat(tables, ignore_index=True)
temperatures.to_csv(RAW / "temperatures_cities.csv", index=False)
print("Temperatures:", temperatures.shape)

new_york      6879 days
boston        6879 days
philadelphia  6879 days
washington    6879 days
pittsburgh    6879 days
cleveland     6879 days
columbus      6879 days
detroit       6879 days
indianapolis  6879 days
chicago       6879 days
milwaukee     6879 days
minneapolis   6879 days
atlanta       6879 days
dallas        6879 days
houston       6879 days
Temperatures: (103185, 3)
